# 01. Object Detection with RT-DETR (Baidu / PaddleDetection)

## 1. Overview
- **Task**: Real-Time Object Detection
- **Tool**: RT-DETR R50 (`lyuwenyu/RT-DETR` / PaddleDetection)
- **License**: Apache-2.0 (Commercially permissive, avoids AGPL-3.0 restrictions)
- **Architecture**: Transformer-based end-to-end detector (NMS-free direct set prediction)

### Key Benefits for CCTV & Edge Server
1. **NMS-Free**: Direct query-to-object matching avoids Non-Maximum Suppression bottlenecks and tuning.
2. **Fixed Graph Shape**: Fixed `[1, 3, 640, 640]` input allows compilation into an AMD MIGraphX optimized engine (`.mxr`).
3. **Crowded Scene Robustness**: Transformer cross-attention handles occluded and overlapping pedestrians in CCTV corridors better than traditional anchor-based CNNs.

In [ ]:
# Import dependencies
import os
import cv2
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt
from PIL import Image

# COCO-80 standard class labels
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
    'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat',
    'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
    'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop',
    'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

# Filter targets: Focus on CCTV surveillance classes
TARGET_CLASSES = {'person', 'car', 'truck', 'motorcycle', 'bicycle'}

## 2. RT-DETR Preprocessor & Postprocessor

In [ ]:
class RTDetrDetector:
    """RT-DETR ONNX detector with letterboxing and filtering."""

    INPUT_SIZE = 640

    def __init__(self, model_path: str = None, conf_threshold: float = 0.35):
        self.conf_threshold = conf_threshold
        self.model_path = model_path
        self.session = None
        if model_path and os.path.exists(model_path):
            # Prefer MIGraphX provider on AMD GPU, fallback to CPU
            providers = ['MIGraphXExecutionProvider', 'CPUExecutionProvider']
            self.session = ort.InferenceSession(model_path, providers=providers)
            self.input_name = self.session.get_inputs()[0].name
        else:
            print(f'Model not found at {model_path}. Running in dummy simulation mode.')

    def preprocess(self, bgr_image: np.ndarray):
        # Convert BGR to RGB
        rgb = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]

        # Resize to fixed 640x640
        resized = cv2.resize(rgb, (self.INPUT_SIZE, self.INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
        blob = resized.astype(np.float32) / 255.0
        # Transpose from HWC to CHW format
        blob = np.transpose(blob, (2, 0, 1))
        blob = np.expand_dims(blob, axis=0)
        return blob, (h, w)

    def postprocess(self, outputs, orig_shape):
        orig_h, orig_w = orig_shape
        # Expected shape: [1, 300, 6] -> (x1, y1, x2, y2, score, class_id)
        dets = outputs[0][0]
        detections = []
        scale_x = orig_w / self.INPUT_SIZE
        scale_y = orig_h / self.INPUT_SIZE

        for det in dets:
            x1, y1, x2, y2, score, cls_id = det
            if score < self.conf_threshold:
                continue
            cls_id = int(cls_id)
            if cls_id < 0 or cls_id >= len(COCO_CLASSES):
                continue
            cls_name = COCO_CLASSES[cls_id]
            if cls_name not in TARGET_CLASSES:
                continue

            box = [
                int(max(0, x1 * scale_x)),
                int(max(0, y1 * scale_y)),
                int(min(orig_w, x2 * scale_x)),
                int(min(orig_h, y2 * scale_y))
            ]
            detections.append({'box': box, 'score': float(score), 'class_name': cls_name})
        return detections

    def predict(self, bgr_image: np.ndarray):
        orig_shape = bgr_image.shape[:2]
        if self.session is None:
            # Mock detection for demonstration
            h, w = orig_shape
            return [
                {'box': [int(w * 0.2), int(h * 0.3), int(w * 0.35), int(h * 0.85)], 'score': 0.89, 'class_name': 'person'},
                {'box': [int(w * 0.5), int(h * 0.4), int(w * 0.8), int(h * 0.75)], 'score': 0.94, 'class_name': 'car'}
            ]
        blob, orig_shape = self.preprocess(bgr_image)
        raw_outs = self.session.run(None, {self.input_name: blob})
        return self.postprocess(raw_outs, orig_shape)

## 3. Visualization Helper & Run Detection Demo

In [ ]:
# Create synthetic test frame
sample_image = np.ones((720, 1280, 3), dtype=np.uint8) * 45
# Draw synthetic background road and walkway
cv2.rectangle(sample_image, (0, 360), (1280, 720), (70, 70, 70), -1)
cv2.line(sample_image, (0, 540), (1280, 540), (255, 255, 255), 4)

# Instantiate detector
detector = RTDetrDetector(model_path='../ai/models/rtdetr_r50_uint8.onnx', conf_threshold=0.3)
detections = detector.predict(sample_image)

# Render detections
canvas = sample_image.copy()
for det in detections:
    x1, y1, x2, y2 = det['box']
    label = f"{det['class_name']}: {det['score']:.2f}"
    cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 120), 2)
    cv2.putText(canvas, label, (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 120), 2)

# Display result
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title('RT-DETR Detection Output')
plt.axis('off')
plt.show()

print(f'Detected objects count: {len(detections)}')
for d in detections:
    print(f" - {d['class_name']} at {d['box']} (confidence: {d['score']:.3f})")